# Iterative Prompt Development: Principles, Mechanics & Practice

In real-world applications with Large Language Models (LLMs), initial prompts rarely produce the exact final production-ready output on the very first attempt. 

Effective prompt engineering is fundamentally an **iterative feedback loop** akin to traditional machine learning model development:

```
   ┌───────────────────────────────────────────┐
   │                  IDEA                     │
   │  (Define clear task, role, context, goal) │
   └─────────────────────┬─────────────────────┘
                         │
                         ▼
   ┌───────────────────────────────────────────┐
   │             IMPLEMENTATION                │
   │  (Write prompt with delimiters/constraints│
   └─────────────────────┬─────────────────────┘
                         │
                         ▼
   ┌───────────────────────────────────────────┐
   │             EXPERIMENTAL RESULT           │
   │  (Run completion, inspect output/length)  │
   └─────────────────────┬─────────────────────┘
                         │
                         ▼
   ┌───────────────────────────────────────────┐
   │               ERROR ANALYSIS              │
   │ (Diagnose failure: too long? wrong audience?│
   │  missing entities? unformatted data?)     │
   └─────────────────────┬─────────────────────┘
                         │
                         └──────► (Refine & repeat)
```

Rather than seeking an elusive "perfect static prompt", master the **process of iterative refinement**.

## 1. Environment Setup & Configuration

Configure credentials and initialize the client.
- Supports standard OpenAI API credentials (`OPENAI_API_KEY`).
- Supports Google Gemini via its OpenAI-compatible endpoint (`GEMINI_API_KEY` with `https://generativelanguage.googleapis.com/v1beta/`).

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

# Load secret API key from the local .env file
_ = load_dotenv(find_dotenv())

# Automatic configuration for OpenAI or Gemini endpoint
api_key = os.getenv("OPENAI_API_KEY") or os.getenv("GEMINI_API_KEY")
base_url = "https://generativelanguage.googleapis.com/v1beta/" if os.getenv("GEMINI_API_KEY") and not os.getenv("OPENAI_API_KEY") else None

if base_url:
    client = OpenAI(api_key=api_key, base_url=base_url)
else:
    client = OpenAI(api_key=api_key)

### Helper Function: `get_completion`
Encapsulates chat completion requests. `temperature=0` enforces deterministic, reproducible outputs.

In [ ]:
def get_completion(prompt, model="gpt-3.5-turbo"):
    """
    Helper wrapper to execute single-turn user prompts.
    If using Gemini, model can be set to 'gemini-2.5-flash', 'gemini-2.0-flash', etc.
    """
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
    )
    return response.choices[0].message.content

## 2. The Case Study: Technical Product Fact Sheet

Throughout this notebook, we transform an unstructured, multi-section engineering fact sheet for an office chair into customer-facing marketing collateral.

In [ ]:
fact_sheet_chair = """
OVERVIEW
- Part of a beautiful family of mid-century inspired office furniture, 
including filing cabinets, desks, bookcases, meeting tables, and more.
- Several options of shell color and base finishes.
- Available with plastic back and front upholstery (SWC-100) 
or full upholstery (SWC-110) in 10 fabric and 6 leather options.
- Base finish options are: stainless steel, matte black, 
gloss white, or chrome.
- Chair is available with or without armrests.
- Suitable for home or business settings.
- Qualified for contract use.

CONSTRUCTION
- 5-wheel plastic coated aluminum base.
- Pneumatic chair adjust for easy raise/lower action.

DIMENSIONS
- WIDTH 53 CM | 20.87”
- DEPTH 51 CM | 20.08”
- HEIGHT 80 CM | 31.50”
- SEAT HEIGHT 44 CM | 17.32”
- SEAT DEPTH 41 CM | 16.14”

OPTIONS
- Soft or hard-floor caster options.
- Two choices of seat foam densities: 
 medium (1.8 lb/ft3) or high (2.8 lb/ft3)
- Armless or 8 position PU armrests 

MATERIALS
SHELL BASE GLIDER
- Cast Aluminum with modified nylon PA6/PA66 coating.
- Shell thickness: 10 mm.
SEAT
- HD36 foam

COUNTRY OF ORIGIN
- Italy
"""

## 3. Iteration 1: Initial Prompt Draft

### Step 1: Idea & Implementation
Establish the baseline task: instruct the model to write marketing copy from the fact sheet using delimiters (` ``` `) to isolate technical specifications.

In [ ]:
prompt_v1 = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

Technical specifications: ```{fact_sheet_chair}```
"""

response_v1 = get_completion(prompt_v1)
print("--- Iteration 1 Response ---")
print(response_v1)

### Step 2: Error Analysis on Iteration 1
- **Observation:** The generated text is well-written and covers the product, but it is **excessively verbose** for an e-commerce catalog or mobile product card.
- **Diagnosis:** The prompt lacked an explicit length constraint.

## 4. Iteration 2: Constraining Output Length

### Strategy: Limiting Output Scope
You can guide length using words, sentences, or characters:
1. `Use at most 50 words.` (Recommended for general conciseness)
2. `Use at most 3 sentences.` (Enforces structural conciseness)
3. `Use at most 280 characters.` (Character limits are approximate because LLMs process **sub-word tokens**, not raw characters directly).

In [ ]:
prompt_v2 = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

Use at most 50 words.

Technical specifications: ```{fact_sheet_chair}```
"""

response_v2 = get_completion(prompt_v2)
print("--- Iteration 2 Response ---")
print(response_v2)
print("\nWord count:", len(response_v2.split()))

### Step 2: Error Analysis on Iteration 2
- **Observation:** Length is now concise (~50 words).
- **New Business Constraint / Diagnosis:** The copy is tailored to direct consumers (B2C). However, the business is selling wholesale to **furniture retailers (B2B)**, who require technical material specs rather than decorative lifestyle descriptions.

## 5. Iteration 3: Aligning Audience & Adding Specific Entity Extraction

### Step 1: Implementation
1. **Specify Target Audience & Tone:** Reorient copy to furniture retailers focusing on materials and construction.
2. **Entity Extraction Constraint:** Demand that all 7-character Product IDs (e.g., `SWC-100`, `SWC-110`) from the fact sheet appear at the end of the text.

In [ ]:
prompt_v3 = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

The description is intended for furniture retailers, 
so should be technical in nature and focus on the 
materials the product is constructed from.

At the end of the description, include every 7-character 
Product ID in the technical specification.

Use at most 50 words.

Technical specifications: ```{fact_sheet_chair}```
"""

response_v3 = get_completion(prompt_v3)
print("--- Iteration 3 Response ---")
print(response_v3)

### Step 2: Error Analysis on Iteration 3
- **Observation:** The copy focuses on aluminum coatings and foam densities and successfully extracts `SWC-100` and `SWC-110`.
- **New Frontend Requirement:** The web development team needs structured dimension specifications in an HTML table, with measurements converted strictly to inches, encapsulated in a `<div>` element for direct website rendering.

## 6. Iteration 4: Multi-Step Structured HTML Table Generation

### Step 1: Implementation
Add explicit structural and data transformation constraints:
- Include a 2-column table titled `'Product Dimensions'`.
- Column 1: Dimension name.
- Column 2: Measurement in inches only.
- Wrap the description in a `<div>` and format all output as valid HTML.

In [ ]:
prompt_v4 = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

The description is intended for furniture retailers, 
so should be technical in nature and focus on the 
materials the product is constructed from.

At the end of the description, include every 7-character 
Product ID in the technical specification.

After the description, include a table that gives the 
product's dimensions. The table should have two columns.
In the first column include the name of the dimension. 
In the second column include the measurements in inches only.

Give the table the title 'Product Dimensions'.

Format everything as HTML that can be used in a website. 
Place the description in a <div> element.

Technical specifications: ```{fact_sheet_chair}```
"""

response_v4 = get_completion(prompt_v4)
print("--- Raw Generated HTML Output ---")
print(response_v4)

### Rendering and Inspecting the HTML Output
Using IPython's `display` and `HTML` tools, verify that the markup renders correctly.

In [ ]:
from IPython.display import display, HTML

# Render the generated markup directly within the notebook cell
display(HTML(response_v4))

## 7. Scaling the Iterative Process: From 1 Example to Evaluation Datasets

### Early Stage vs. Mature Application Development

| Stage | Development Methodology | Evaluation Approach |
| :--- | :--- | :--- |
| **Early Prototyping** | Fast interactive testing on 1 to 3 representative samples | Manual visual inspection & prompt tweaking |
| **Production / Mature System** | Benchmark testing against 50 to 500+ golden evaluation cases | Automated metrics (e.g., entity recall, schema validation, PR-AUC, exact-match, LLM-as-a-judge) |

In [ ]:
# Demonstrating automated evaluation across multiple test fact sheets
test_fact_sheets = [
    {"id": "chair_01", "text": fact_sheet_chair, "expected_ids": ["SWC-100", "SWC-110"]}
]

def evaluate_prompt(prompt_template, dataset):
    scores = []
    for sample in dataset:
        test_prompt = prompt_template.format(specs=sample["text"])
        result = get_completion(test_prompt)
        
        # Check if all required IDs were extracted
        ids_present = all(pid in result for pid in sample["expected_ids"])
        scores.append(int(ids_present))
        
    accuracy = sum(scores) / len(scores)
    return f"Entity Extraction Accuracy: {accuracy * 100:.1f}%"

# Sample evaluation execution
prompt_template = """
Summarize materials and list all 7-character product IDs from specs: ```{specs}```
"""
print("Evaluation Result:", evaluate_prompt(prompt_template, test_fact_sheets))

## 8. Key Takeaways & Cheat Sheet

1. **Prompt Engineering is an Empirical Loop:** No prompt is universally perfect from the start. Formulate an idea $\rightarrow$ implement the prompt $\rightarrow$ evaluate experimental output $\rightarrow$ conduct error analysis $\rightarrow$ iterate.
2. **Diagnose Before Modifying:**
   - Output too verbose? $\rightarrow$ Enforce word or sentence constraints.
   - Wrong tone/details? $\rightarrow$ Clarify audience and prioritized fields.
   - Missing critical data? $\rightarrow$ Mandate entity extraction rules.
   - Hard to parse? $\rightarrow$ Enforce structured schemas (JSON, HTML tables).
3. **Tokenization Awareness:** Models count sub-word tokens rather than raw characters or syllables. Word and sentence limits are followed more reliably than exact character counts.
4. **Shift from Heuristics to Metrics:** Transition from manual single-example prompt adjustments to automated batch testing over golden benchmark sets as applications approach production.